# Train Impact AI

We have 3 actions:
- 0 wait
- 1 horse
- 2 tower

In [15]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import requests


class TowerDefenceEnv(gym.Env):
    metadata = {}

    def __init__(self, unity_url="http://127.0.0.1:8000"):
        super().__init__()

        self.unity_url = unity_url
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=-1.0, high=1.0, shape=(64,), dtype=np.float32)
        self.previous_state = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        response = requests.get(f"{self.unity_url}/game-state")
        state = response.json()
        print("STATE:", state)
        self.previous_state = state
        observation = self.state_to_obs(state)
        return observation, {}

    def step(self, action):
        response = requests.post(
            f"{self.unity_url}/step",
            json={ "action": int(action) }
        )

        state = response.json()
        reward = self.calculate_reward(self.previous_state, state, action)
        terminated = bool(state["game_finished"])
        truncated = False
        observation = self.state_to_obs(state)
        self.previous_state = state
        info = {"winner_player_id": state.get("winner_player_id", -1)}
        return observation, reward, terminated, truncated, info

    def state_to_obs(self, state):
        obs = []
        coins = state["coins"]
        obs.append(np.clip(coins / 100.0, 0.0, 1.0))

        own_hp = state.get("own_castle_hp", 100)
        enemy_hp = state.get("enemy_castle_hp", 100)
        obs.append(own_hp / 100.0)
        obs.append(enemy_hp / 100.0)

        enemies = state["enemy_units"]
        for enemy in enemies[:20]:

            if enemy["exists"]:
                obs.append(np.clip(enemy["x"] / 20.0, -1, 1))
                obs.append(np.clip(enemy["y"] / 20.0, -1, 1))
                obs.append(1.0)
            else:
                obs.extend([0.0, 0.0, 0.0])

        while len(enemies) < 20:
            obs.extend([0.0, 0.0, 0.0])
            enemies.append({"exists": False})

        available_positions = (state["available_tower_positions"])
        obs.append(min(len(available_positions) / 50.0, 1.0))

        return np.asarray(obs, dtype=np.float32)

In [16]:
env = TowerDefenceEnv()

obs, info = env.reset()
print(obs)

# obs, reward, terminated, truncated, info = env.step(1)
# print(obs, reward)

STATE: {'ready': False}


KeyError: 'coins'